Original dataset file: "original datasets\UCI_Credit_Card.csv"

In [3]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"original datasets\UCI_Credit_Card.csv")

In [5]:
print(df.shape)

(30000, 25)


Matches the shape of the original dataset.

In [7]:
print(df.info())


<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          30000 non-null  int64  
 1   LIMIT_BAL                   30000 non-null  float64
 2   SEX                         30000 non-null  int64  
 3   EDUCATION                   30000 non-null  int64  
 4   MARRIAGE                    30000 non-null  int64  
 5   AGE                         30000 non-null  int64  
 6   PAY_0                       30000 non-null  int64  
 7   PAY_2                       30000 non-null  int64  
 8   PAY_3                       30000 non-null  int64  
 9   PAY_4                       30000 non-null  int64  
 10  PAY_5                       30000 non-null  int64  
 11  PAY_6                       30000 non-null  int64  
 12  BILL_AMT1                   30000 non-null  float64
 13  BILL_AMT2                   30000 non-null

In [8]:
print(df.dtypes)

ID                              int64
LIMIT_BAL                     float64
SEX                             int64
EDUCATION                       int64
MARRIAGE                        int64
AGE                             int64
PAY_0                           int64
PAY_2                           int64
PAY_3                           int64
PAY_4                           int64
PAY_5                           int64
PAY_6                           int64
BILL_AMT1                     float64
BILL_AMT2                     float64
BILL_AMT3                     float64
BILL_AMT4                     float64
BILL_AMT5                     float64
BILL_AMT6                     float64
PAY_AMT1                      float64
PAY_AMT2                      float64
PAY_AMT3                      float64
PAY_AMT4                      float64
PAY_AMT5                      float64
PAY_AMT6                      float64
default.payment.next.month      int64
dtype: object


All columns have appropriate data types.

Columns SEX, EDUCATION, MARRIAGE are integer types:

SEX: Gender (1=male, 2=female)
EDUCATION: (1=graduate school, 2=university, 3=high school, 4=others, 5=unknown, 6=unknown)
MARRIAGE: Marital status (1=married, 2=single, 3=others)
(source: https://www.kaggle.com/datasets/uciml/default-of-credit-card-clients-dataset?resource=download)

In [11]:
print(df.head())

   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  \
0   1    20000.0    2          2         1   24      2      2     -1     -1   
1   2   120000.0    2          2         2   26     -1      2      0      0   
2   3    90000.0    2          2         2   34      0      0      0      0   
3   4    50000.0    2          2         1   37      0      0      0      0   
4   5    50000.0    1          2         1   57     -1      0     -1      0   

   ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  \
0  ...        0.0        0.0        0.0       0.0     689.0       0.0   
1  ...     3272.0     3455.0     3261.0       0.0    1000.0    1000.0   
2  ...    14331.0    14948.0    15549.0    1518.0    1500.0    1000.0   
3  ...    28314.0    28959.0    29547.0    2000.0    2019.0    1200.0   
4  ...    20940.0    19146.0    19131.0    2000.0   36681.0   10000.0   

   PAY_AMT4  PAY_AMT5  PAY_AMT6  default.payment.next.month  
0       0.0       0.0   

In [12]:
print(df.isnull().sum())

ID                            0
LIMIT_BAL                     0
SEX                           0
EDUCATION                     0
MARRIAGE                      0
AGE                           0
PAY_0                         0
PAY_2                         0
PAY_3                         0
PAY_4                         0
PAY_5                         0
PAY_6                         0
BILL_AMT1                     0
BILL_AMT2                     0
BILL_AMT3                     0
BILL_AMT4                     0
BILL_AMT5                     0
BILL_AMT6                     0
PAY_AMT1                      0
PAY_AMT2                      0
PAY_AMT3                      0
PAY_AMT4                      0
PAY_AMT5                      0
PAY_AMT6                      0
default.payment.next.month    0
dtype: int64


No null values found in any column.

To check for duplicates, first checking if every entry has a unique ID. 
Then, drop the ID column to see if any entries have completely identical features (duplicates).

In [13]:
ids = pd.Series(df["ID"])
print(ids.is_unique)

True


All IDs are unique

In [26]:
print(df.duplicated(subset=df.columns.drop("ID")).sum())

35


35 duplicates found.

In [34]:
cols = df.columns.drop("ID")
dups = df[df.duplicated(subset=cols, keep=False)].sort_values(list(cols))
bill_cols = [c for c in df.columns if c.startswith('BILL_AMT')]
pay_amt_cols = [c for c in df.columns if c.startswith('PAY_AMT')]
inactive = (dups[bill_cols + pay_amt_cols] == 0).all(axis=1)
print(dups[inactive]["default.payment.next.month"].value_counts())
print(inactive.sum(), (~inactive).sum(), len(dups))

default.payment.next.month
0    50
1    10
Name: count, dtype: int64
60 10 70


In [36]:
group_sizes = dups.groupby(list(cols)).size()
print(group_sizes.value_counts())

2    35
Name: count, dtype: int64


**Duplicates:** All IDs are unique, but 70 rows form 35 pairs that are identical across every column except ID.
Of these, 60 rows (30 pairs) have zero bills and zero payments in all months ("inactive"), so matching could be coincidence.
The other 10 rows (5 pairs) have non-trivial histories and are more likely true duplicates.
Handling is decided in 02_data_cleaning.ipynb.

In [35]:
print(df["SEX"].value_counts())
print(df["MARRIAGE"].value_counts())
print(df["EDUCATION"].value_counts())
print(df["PAY_0"].value_counts())
print(df["PAY_2"].value_counts())
print(df["PAY_3"].value_counts())
print(df["PAY_4"].value_counts())
print(df["PAY_5"].value_counts())
print(df["PAY_6"].value_counts())

SEX
2    18112
1    11888
Name: count, dtype: int64
MARRIAGE
2    15964
1    13659
3      323
0       54
Name: count, dtype: int64
EDUCATION
2    14030
1    10585
3     4917
5      280
4      123
6       51
0       14
Name: count, dtype: int64
PAY_0
 0    14737
-1     5686
 1     3688
-2     2759
 2     2667
 3      322
 4       76
 5       26
 8       19
 6       11
 7        9
Name: count, dtype: int64
PAY_2
 0    15730
-1     6050
 2     3927
-2     3782
 3      326
 4       99
 1       28
 5       25
 7       20
 6       12
 8        1
Name: count, dtype: int64
PAY_3
 0    15764
-1     5938
-2     4085
 2     3819
 3      240
 4       76
 7       27
 6       23
 5       21
 1        4
 8        3
Name: count, dtype: int64
PAY_4
 0    16455
-1     5687
-2     4348
 2     3159
 3      180
 4       69
 7       58
 5       35
 6        5
 1        2
 8        2
Name: count, dtype: int64
PAY_5
 0    16947
-1     5539
-2     4546
 2     2626
 3      178
 4       84
 7       58
 5       1

Gender: 1=male, 2=female
Marriage: 1=married, 2=single, 3=others, 0=undocumented (count=54)
Education: 1=graduate school, 2=university, 3=high school, 4=others, 5=unknown, 6=unknown, 0=unkown
    5,6,0 are undocumented (total count=335)
Pay_columns (from 0-6): -1=pay duly, 1=payment delay for one month, 2=payment delay for two months, … 8=payment delay for eight months -2=undocumented 0=undocumented (these 2 columns account for over 50% of the data in these columns)

In [24]:
print(df["default.payment.next.month"].value_counts(normalize=True))

default.payment.next.month
0    0.7788
1    0.2212
Name: proportion, dtype: float64


About 22% of users defaulted on their next month payment.

In [27]:
id_cols = ['ID']
demo_cols = ['SEX', 'EDUCATION', 'MARRIAGE', 'AGE']
credit_cols = ['LIMIT_BAL']
repay_cols = [c for c in df.columns if c.startswith('PAY_') and not c.startswith('PAY_AMT')]
bill_cols = [c for c in df.columns if c.startswith('BILL_AMT')]
pay_amt_cols = [c for c in df.columns if c.startswith('PAY_AMT')]
target_col = ['default.payment.next.month']

all_grouped = id_cols + demo_cols + credit_cols + repay_cols + bill_cols + pay_amt_cols + target_col
print(len(all_grouped), len(df.columns), set(all_grouped) == set(df.columns))

25 25 True


Seperated identifier, demographic, credit, repayment, bill, pay amount and target columns.

## Data Dictionary

| Column / Group | Group | Meaning | Type | Notes |
|---|---|---|---|---|
| `ID` | Identifier | Unique row identifier | Identifier | Not a feature; dropped before modelling |
| `LIMIT_BAL` | Credit | Credit limit given to the client (NT$), including supplementary credit | Numeric | |
| `SEX` | Demographic | 1 = male, 2 = female | Categorical | |
| `EDUCATION` | Demographic | 1 = graduate school, 2 = university, 3 = high school, 4 = others | Categorical | 0, 5, 6 are undocumented (335 rows total) |
| `MARRIAGE` | Demographic | 1 = married, 2 = single, 3 = others | Categorical | 0 is undocumented (54 rows) |
| `AGE` | Demographic | Age in years | Numeric | Range to be validated in cleaning |
| `PAY_0` | Repayment | Repayment status, most recent month | Ordinal | See PAY codes below |
| `PAY_2` ... `PAY_6` | Repayment | Repayment status, 2 to 6 months back | Ordinal | No `PAY_1` exists; `PAY_0` is the most recent month |
| `BILL_AMT1` ... `BILL_AMT6` | Bill | Bill statement amount (what was owed), `BILL_AMT1` = most recent | Numeric | Negative values to be investigated in cleaning |
| `PAY_AMT1` ... `PAY_AMT6` | Payment | Amount actually paid, `PAY_AMT1` = most recent | Numeric | |
| `default.payment.next.month` | Target | 1 = defaulted next month, 0 = did not | Binary | About 22% are 1 (class imbalance) |

**PAY code values:** -1 = paid duly, 1 to 8 = payment delayed by that many months. Codes `-2` and `0` are not in the documentation and make up over 50% of values in these columns, so they need a decision in `02_data_cleaning.ipynb`.